# Tutorial 18: linear-light scattering from a Co film with a magnon-like modulation

A **30 nm Co film at normal incidence**, illuminated by **linear X-rays at 778 eV** in the Co L-edge region. The film is mainly magnetized in plane, with an editable sinusoidal out-of-plane component of **40 nm period and 10% amplitude**. Follow the complex exit wave → FFT → a physical flat-detector pattern, then insert an ideal linear analyzer immediately before the detector.

Compare **no analyzer**, **parallel**, an **editable analyzer axis**, and **crossed** (total extinction relative to the original illumination). All intensity comparisons retain one common scale; the weak crossed signal is never normalized up to the unfiltered brightness.

This is a frozen spatial snapshot resembling a magnon wave, not a calculation of magnon dynamics or energy-resolved inelastic scattering. The ideal analyzer has unit transmission along its pass axis and zero transmission along the orthogonal axis. Camera noise and finite analyzer leakage are omitted to expose the optical effect.


In [ ]:
from pathlib import Path
from dataclasses import replace
import sys, json, hashlib, platform, subprocess
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/scattering_calculator").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from scattering_calculator.database.database_loading import material_params
from scattering_calculator import simulation_pipelines as sim
from scattering_calculator.simulation_pipelines.experiment import analyze_jones
from scattering_calculator.experimental_conditions import light_beam, detector

OUT = ROOT / "outputs" / "18_co_magnon_linear_analyzer"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.constrained_layout.use"] = True

def save_figure(fig, name):
    for suffix in ("png", "pdf"):
        fig.savefig(OUT / f"{name}.{suffix}", dpi=160, bbox_inches="tight")
    plt.show()


## 1. Editable sample, illumination and detector parameters

Angles here are measured **counterclockwise from laboratory +x toward +y**. The incident electric field is horizontal (+x) by default. The mean magnetization is also along +x; change `magnetic_pattern.pattern_config["inplane_angle"]` independently. The modulation runs along the direction `magnetic_pattern.pattern_config["angle"]` in the film plane. `amplitude=0.10` means a peak perpendicular component of 10% of saturation, **not** a 10% increase in total magnetization.

A finite Gaussian beam illuminates an otherwise uniform film. Its amplitude is `exp(-r²/(2 sigma²))`. There is no FTH mask or reference hole. The grid and beam width determine the illuminated number of periods and satellite width. Dimensions in the configuration are explicitly nm or m. Run All after editing.

### Rotation controls — edit the three values in the next cell

| Control | What rotates |
| --- | --- |
| `xray.linear_polarization_angle` | Linear electric-field axis in the sample plane; the beam still propagates normally along +z |
| `magnetic_pattern.pattern_config["angle"]` | Magnon wavevector in the film plane; the stripes lie perpendicular to it, and scattering satellites rotate with it |
| `detector.analyzer_angle` | Pass axis of the final polarizer in the laboratory frame |

All three are independent angles in radians, measured from +x toward +y. For
example, use 30°, 60°, and 120° to rotate the illumination, wavevector, and
polarizer separately. The extra **parallel** and **crossed** comparison axes
always follow the incident polarization automatically. To make the editable
polarizer crossed, set `analyzer_angle = illumination_angle + np.pi/2`.
Changing `magnetic_pattern.pattern_config["inplane_angle"]` separately rotates the mean in-plane magnetization.
After editing, **Run All** to regenerate the exit waves, FFT and detector images.


## Standard experiment setup

Use the same production configuration sections in every example:
**X-ray source → simulation grid → sample → magnetization → apertures →
illumination → sample propagation → detector/acquisition → beamstop → energy scan**.
Physical lengths are metres, energy is eV, and angles are radians; recipe
thickness literals are nm. See [the setup standard](../docs/experiment_setup.md).
Specialized plotting, reconstruction and analytic-reference settings are analysis
controls, separate from the physical experiment definition.


In [ ]:
# All physical lengths are metres; energy eV; rotation angles radians.
illumination_angle = np.deg2rad(0.)
wave_angle = np.deg2rad(0.)
analyzer_angle = np.deg2rad(45.)
# For crossed extinction: analyzer_angle = illumination_angle + np.pi/2
experiment = sim.ExperimentConfig(
    xray=sim.XRayConfig(energy=778., photon_flux=1., pol="LH",
                       coherence_length=(1.,1.), linear_polarization_angle=illumination_angle),
    simulation=sim.SimulationConfig(shape=(384,384), real_space_pixel_size=2e-9),
    sample=sim.SampleConfig(recipe="Co(30)",max_slice_thickness=2e-9,sample_name="Co magnon-like film"),
    magnetic_pattern=sim.MagneticPatternConfig(pattern_type_method="magnon_wave",pattern_config={
        "period":40e-9, 'amplitude':.1, 'angle':wave_angle,
        "inplane_angle":0., "phase":0.,
    }),
    aperture=sim.FrontApertureConfig(aperture_method=None),
    illumination=sim.IlluminationConfig(illumination_function="gaussian",illumination_config={
        "center":(0.,0.),"distance":0.,"fwhm":2*np.sqrt(np.log(2))*90e-9,
        "alpha_beam":(0.,0.),
    }),
    propagation=sim.SamplePropagatorConfig(propagator_method="Jones",propagator_config={
        "propagate":True,"calculate_farfield":False,"farfield_oversampling":4,
    }),
    detector=sim.DetectorConfig(shape=(256,256),pixel_size=13.5e-6,
        sample_to_detector_distance=.03,detector_center=(128,128),
        detector_propagation_method="fraunhofer",analyzer_angle=analyzer_angle,
        use_detector_pixel_footprint=True,detector_pixel_footprint_samples=3,
        detector_params={"counts_per_photon":1.,"quantum_efficiency":1.,
            "readout_noise_average":0.,"readout_noise_sigma":0.,"detector_threshold":1e12},
        artifacts_config={"sigma_photon":0.},
        measurement_config={"exposure_time":1.,"number_frames":1,"max_counts_per_image":None}),
    beamstop=sim.BeamstopConfig(bs_method=None),
    energies_eV=(778.,),
)
print(json.dumps(experiment.to_dict(),indent=2))


## 2. Construct the magnetization and optical response

For the in-plane wave coordinate $s=x\cos\theta_w+y\sin\theta_w$,

$$m_z=A\sin(2\pi s/\Lambda+\phi),\qquad
(m_x,m_y)=\sqrt{1-m_z^2}(\cos\theta_m,\sin\theta_m).$$

Thus $|\mathbf m|=1$ everywhere. The same magnetization is repeated through the thickness. Co is split into slices no thicker than `sample.max_slice_thickness`. The loader returns charge, circular and linear channels `(n0, nc, nl)` at the chosen energy. The bundled Co example has **zero linear channel**: this experiment probes the polarization conversion due to its circular magnetic response and the OOP modulation; it does not demonstrate XMLD.


In [ ]:
grid = experiment.simulation
grid.setup()
x_nm, y_nm = grid.xgrid*1e9, grid.ygrid*1e9
u_nm = x_nm[0]
experiment.magnetic_pattern.shape = grid.shape
experiment.magnetic_pattern.real_space_pixel_size = grid.real_space_pixel_size
m, _ = experiment.magnetic_pattern.create_pattern()
mz = m[...,2]
wave_angle = experiment.magnetic_pattern.pattern_config['angle']
inplane_angle = experiment.magnetic_pattern.pattern_config['inplane_angle']
np.testing.assert_allclose(np.linalg.norm(m,axis=-1),1.,atol=1e-14)

materials = material_params(materials=["Co"], x_ray_energy=experiment.xray.energy)
optical_channels = material_params.load_refractive_index("Co", experiment.xray.energy)
print("Co [n0, nc, nl]:", optical_channels)
print("Wavelength (nm):", 1239.8419843320026/experiment.xray.energy)
print("Pixels per period:", (experiment.magnetic_pattern.pattern_config['period']*1e9)/(experiment.simulation.real_space_pixel_size*1e9))
print("Gaussian amplitude at grid edge:", np.exp(-((experiment.simulation.shape[0]/2)*(experiment.simulation.real_space_pixel_size*1e9))**2/(2*(experiment.illumination.illumination_config['fwhm']/(2*np.sqrt(np.log(2)))*1e9)**2)))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
extent_nm = (u_nm[0],u_nm[-1],u_nm[0],u_nm[-1])
for ax, component, title in zip(axes, (m[...,0],m[...,1],mz), ("mx", "my", "mz: OOP wave")):
    im = ax.imshow(component, origin="lower", extent=extent_nm, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set(title=title, xlabel="x (nm)", ylabel="y (nm)")
    fig.colorbar(im, ax=ax, label="M / Ms")
save_figure(fig, "01_magnetization")
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(u_nm, mz[experiment.simulation.shape[0]//2], label="mz")
ax.plot(u_nm, m[experiment.simulation.shape[0]//2,:,0], label="mx")
ax.set(xlabel="x (nm), y=0", ylabel="M / Ms", title="Magnetization line through the film")
ax.legend()
save_figure(fig, "01b_magnetization_line")


## 3. Jones multislice: the complex exit wave

Jones propagation is necessary to see light converted into the orthogonal polarization. A scalar intensity calculation followed by a Malus-law multiplier would miss this signal. `propagation.propagator_config={"propagate": False}` disables transverse propagation between slices while retaining material interaction.

We also simulate **the identical film with zero OOP modulation**. This supplies a crossed-analyzer extinction control and a reference exit field. The plotted phases are relative to the central parallel component of that reference, removing an irrelevant common propagation phase.


In [ ]:
incident_angle = experiment.xray.linear_polarization_angle
p_in = np.array([np.cos(incident_angle),np.sin(incident_angle)])
p_cross = np.array([-p_in[1],p_in[0]])

# This setup/propagation sequence is the same in every complete experiment.
run = sim.ScatteringExperiment(experiment).setup(magnetization=m)
beam = run.config.illumination.beam_params
E_in = run.config.illumination.illumination.illumination_jones.copy()

def simulate_exit(magnetization):
    trial = sim.ScatteringExperiment(experiment).setup(magnetization=magnetization)
    return trial.propagate().exit_wave.copy()

exit_wave = simulate_exit(m)
unmodulated_m = np.zeros_like(m)
unmodulated_m[...,0] = np.cos(inplane_angle)
unmodulated_m[...,1] = np.sin(inplane_angle)
reference_exit = simulate_exit(unmodulated_m)
parallel_exit = np.einsum("...c,c->...", exit_wave, p_in)
crossed_exit = np.einsum("...c,c->...", exit_wave, p_cross)
reference_parallel = np.einsum("...c,c->...", reference_exit, p_in)
phase_reference = np.angle(reference_parallel[experiment.simulation.shape[0]//2,experiment.simulation.shape[0]//2])

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for row, (field, title) in enumerate(((parallel_exit,"Parallel to incident E"), (crossed_exit,"Orthogonal to incident E"))):
    amplitude = abs(field)/np.max(abs(E_in))  # relative to incident peak
    im = axes[row,0].imshow(amplitude, origin="lower", extent=extent_nm, cmap="magma")
    axes[row,0].set(title=title+": amplitude", xlabel="x (nm)", ylabel="y (nm)")
    fig.colorbar(im, ax=axes[row,0], label="|E| / incident peak (separate color ranges)")
    phase = np.angle(field*np.exp(-1j*phase_reference))
    # Do not assign a phase to effectively zero-amplitude pixels.
    masked_phase = np.ma.masked_where(amplitude < max(amplitude.max()*1e-4,1e-14), phase)
    im = axes[row,1].imshow(masked_phase, origin="lower", extent=extent_nm, cmap="twilight", vmin=-np.pi,vmax=np.pi)
    axes[row,1].set(title=title+": phase", xlabel="x (nm)", ylabel="y (nm)")
    fig.colorbar(im, ax=axes[row,1], label="phase relative to reference (rad)")
save_figure(fig, "02_exit_wave")
print("Exit wave shape:", exit_wave.shape, "(y, x, Ex/Ey)")
print("Transmitted power / incident power:", np.sum(abs(exit_wave)**2)/np.sum(abs(E_in)**2))
print("Orthogonal exit-power fraction:", np.sum(abs(crossed_exit)**2)/np.sum(abs(exit_wave)**2))


## 4. FFT of the complex field and ideal analyzer

An analyzer with unit pass vector $\mathbf a=(\cos\alpha,\sin\alpha)$ transmits
$E_a=\mathbf a^\dagger\mathbf E$. Its detected intensity is **$|E_a|^2$**, including interference between the complex components. Without an analyzer, intensity is $|E_x|^2+|E_y|^2$.

The analyzer is placed immediately before the detector. Because its axis is uniform and the free-space operator propagates both transverse Jones components identically, applying this projection to the complex far field gives the same result as applying it to the exit field before propagation. This is the simulator's transverse-field approximation; a real wide-angle analyzer needs a calibrated angular response.

Zero padding samples the FFT more finely without adding sample resolution. The plots use the repository's legacy FFT convention. Its transverse momentum orientation is reversed relative to the negative-phase direct RS convention; the symmetric ± satellites in this example are unaffected. The expected first magnetic satellites are at $|q|=2\pi/\Lambda$.


In [ ]:
custom_angle = np.deg2rad(np.rad2deg(experiment.detector.analyzer_angle))
axes_pass = {
    "Parallel": p_in,
    f"Axis {np.rad2deg(experiment.detector.analyzer_angle):g}°": np.array([np.cos(custom_angle),np.sin(custom_angle)]),
    "Crossed": p_cross,
}
labels = ["No analyzer", *axes_pass]

def padded_fft(field):
    total = experiment.simulation.shape[0]*experiment.propagation.propagator_config['farfield_oversampling']
    before = (total-experiment.simulation.shape[0])//2
    after = total-experiment.simulation.shape[0]-before
    padded = np.pad(field, ((before,after),(before,after),(0,0)))
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(padded, axes=(0,1)),
                                      axes=(0,1), norm="ortho"), axes=(0,1))

def analyze_fft(farfield):
    result = {"No analyzer": np.sum(abs(farfield)**2, axis=-1)}
    result.update({label: abs(np.einsum("...c,c->...", farfield, axis))**2
                   for label,axis in axes_pass.items()})
    return result

fft_wave = padded_fft(exit_wave)
fft_reference_wave = padded_fft(reference_exit)
fft_patterns = analyze_fft(fft_wave)
fft_reference_patterns = analyze_fft(fft_reference_wave)
nfft = fft_wave.shape[0]
q_rad_nm = 2*np.pi*np.fft.fftshift(np.fft.fftfreq(nfft, d=(experiment.simulation.real_space_pixel_size*1e9)))
q1 = 2*np.pi/(experiment.magnetic_pattern.pattern_config['period']*1e9)
q_extent = (q_rad_nm[0],q_rad_nm[-1],q_rad_nm[0],q_rad_nm[-1])
q_limit = min(2.4*q1, .95*np.max(q_rad_nm))
fft_scale = fft_patterns["No analyzer"].max()
fig, axes = plt.subplots(1, 4, figsize=(17, 4.1))
for ax,label in zip(axes, labels):
    im = ax.imshow(fft_patterns[label]/fft_scale, origin="lower", extent=q_extent,
                   cmap="magma", norm=LogNorm(1e-12,1), interpolation="nearest")
    ax.set(title=label, xlabel="FFT qx (rad/nm)", ylabel="FFT qy (rad/nm)",
           xlim=(-q_limit,q_limit),ylim=(-q_limit,q_limit))
    for sign in (-1,1):
        ax.plot(sign*q1*np.cos(wave_angle),sign*q1*np.sin(wave_angle),"o",mfc="none",mec="cyan",ms=9)
fig.colorbar(im,ax=list(axes),label="Intensity / unfiltered peak (one common scale)")
save_figure(fig,"03_fft_analyzers")
# Parseval and analyzer completeness: no independent renormalization.
np.testing.assert_allclose(fft_patterns["No analyzer"],
                           fft_patterns["Parallel"]+fft_patterns["Crossed"],rtol=1e-12,atol=1e-15)
np.testing.assert_allclose(np.sum(fft_patterns["No analyzer"]),np.sum(abs(exit_wave)**2),rtol=1e-12)


## 5. Project onto the physical detector

Use Fraunhofer propagation, angular mapping onto a parallel flat detector, and its relative pixel solid-angle factor. `detector.detector_pixel_footprint_samples=3` averages a 3×3 grid inside each pixel. The analyzer has already projected the **complex** far field; the resulting intensities are now sampled onto detector pixels.

These are noise-free relative intensities, not calibrated camera counts. All analyzer patterns use the **same** projection and the **same** display normalization. Black/low-valued crossed pixels mean extinction or weak signal, not a separately rescaled bright image.


In [ ]:
camera = run.config.detector
camera.calc_realspace_resolution(beam)
layout = camera.detector_layout

def detector_projection(patterns):
    projected = {}
    for label,image in patterns.items():
        model = detector.detector_hologram(layout,image,beam,(experiment.simulation.real_space_pixel_size*1e9)*1e-9,None)
        projected[label] = model.gnomonic_projection(
            use_pixel_footprint=experiment.detector.detector_pixel_footprint_samples>1,
            pixel_footprint_samples=experiment.detector.detector_pixel_footprint_samples,
            ignore_flat_detector_curvature=False)
    return projected

patterns = detector_projection(fft_patterns)
reference_patterns = detector_projection(fft_reference_patterns)
detector_scale = patterns["No analyzer"].max()
detector_extent_mm = (
    float(layout.detx.min()*1e3),float(layout.detx.max()*1e3),
    float(layout.dety.min()*1e3),float(layout.dety.max()*1e3))
wavelength_nm = beam.wavelength*1e9
sin_theta = wavelength_nm/(experiment.magnetic_pattern.pattern_config['period']*1e9)
assert sin_theta < 1, "The requested first diffraction order is evanescent."
satellite_radius_m = experiment.detector.sample_to_detector_distance*np.tan(np.arcsin(sin_theta))
print("Expected first-order scattering angle (deg):", np.rad2deg(np.arcsin(sin_theta)))
print("Expected satellite distance from beam center (mm):", satellite_radius_m*1e3)
if (abs(satellite_radius_m*np.cos(wave_angle)) > abs(layout.detx).max()
        or abs(satellite_radius_m*np.sin(wave_angle)) > abs(layout.dety).max()):
    print("First satellites fall outside this detector: increase acceptance or decrease distance.")
fig, axes = plt.subplots(1,4,figsize=(17,4.1))
for ax,label in zip(axes,labels):
    im = ax.imshow(patterns[label]/detector_scale,origin="lower",extent=detector_extent_mm,
                   cmap="magma",norm=LogNorm(1e-12,1),interpolation="nearest")
    ax.set(title=label,xlabel="detector X (mm)",ylabel="detector Y (mm)")
    for sign in (-1,1):
        ax.plot(sign*satellite_radius_m*1e3*np.cos(wave_angle),
                sign*satellite_radius_m*1e3*np.sin(wave_angle),"o",mfc="none",mec="cyan",ms=9)
fig.colorbar(im,ax=list(axes),label="Intensity / unfiltered detector peak (one common scale)")
save_figure(fig,"04_detector_analyzers")
np.testing.assert_allclose(patterns["No analyzer"], patterns["Parallel"]+patterns["Crossed"],rtol=1e-12,atol=1e-14)


## 6. What changes when the analyzer is crossed?

A crossed analyzer extinguishes the **original incident polarization**, not every photon leaving a magnetic film. The OOP magnetic modulation can convert a small part of the linear input into its orthogonal component. With the bundled zero-XMLD model and this geometry, that component has first-order ±1 magnetic satellites; the analyzer suppresses the strong unchanged-polarization central beam. The parallel channel retains the central beam and can contain weaker even-order features. A tilted analyzer also mixes the parallel and converted components coherently.

The zero-modulation control should be extinguished in the crossed channel for the default model. This comparison is more informative than applying `cos²(90°)=0` to the total unfiltered intensity. A uniform analyzer cannot create optical power: parallel plus crossed must equal the no-analyzer signal at every pixel.

The following profiles retain a common scale. The detector line follows the editable wave direction, rather than assuming the satellites lie on a horizontal row. The optional crossed-only panel explicitly uses its own peak to make weak structure visible; use the common-scale panels and printed ratios to assess its brightness.


In [ ]:
from scipy.ndimage import map_coordinates
radius_mm = min(abs(layout.detx).max(),abs(layout.dety).max())*1e3
line_mm = np.linspace(-radius_mm,radius_mm,801)
line_x = line_mm*1e-3*np.cos(wave_angle)/experiment.detector.pixel_size + layout.detector_center[1]
line_y = line_mm*1e-3*np.sin(wave_angle)/experiment.detector.pixel_size + layout.detector_center[0]
fig,ax = plt.subplots(figsize=(11,4.5))
for label in labels:
    line = map_coordinates(patterns[label], [line_y,line_x],order=1,mode="constant",cval=0)
    ax.semilogy(line_mm,np.maximum(line/detector_scale,1e-18),label=label)
control_line = map_coordinates(reference_patterns["Crossed"],[line_y,line_x],order=1,mode="constant",cval=0)
ax.semilogy(line_mm,np.maximum(control_line/detector_scale,1e-18),"--",label="Crossed, no OOP modulation")
for sign in (-1,1): ax.axvline(sign*satellite_radius_m*1e3,color="gray",ls=":")
ax.set(xlabel="Detector distance along wave direction (mm)",ylabel="Intensity / unfiltered peak",
       ylim=(1e-16,2),title="Same-scale detector profiles; dotted lines mark ±1 orders")
ax.legend()
save_figure(fig,"05_detector_profiles")

metrics = {
    "crossed_peak_over_unfiltered_peak": float(patterns["Crossed"].max()/detector_scale),
    "crossed_collected_power_fraction": float(patterns["Crossed"].sum()/patterns["No analyzer"].sum()),
    "crossed_unmodulated_peak_over_unfiltered_peak": float(reference_patterns["Crossed"].max()/detector_scale),
    "first_order_radius_mm": float(satellite_radius_m*1e3),
}
print(json.dumps(metrics,indent=2))
for label,image in patterns.items():
    assert np.isfinite(image).all() and np.all(image >= 0),label
    assert np.all(image <= patterns["No analyzer"] + 1e-12*detector_scale),label
# This control applies to the bundled Co model, whose linear channel is zero.
if abs(optical_channels[2]) == 0:
    assert metrics["crossed_unmodulated_peak_over_unfiltered_peak"] < 1e-20

cross_peak = patterns["Crossed"].max()
if cross_peak > 0:
    fig,axes = plt.subplots(1,2,figsize=(10,4))
    for ax,image,title in zip(axes,(patterns["Crossed"],reference_patterns["Crossed"]),
                             ("Crossed, OOP modulation","Crossed, unmodulated control")):
        im=ax.imshow(image/cross_peak,origin="lower",extent=detector_extent_mm,
                     cmap="magma",norm=LogNorm(1e-6,1))
        ax.set(title=title,xlabel="detector X (mm)",ylabel="detector Y (mm)")
    fig.colorbar(im,ax=list(axes),label="Intensity / modulated CROSSED peak (detail scale only)")
    save_figure(fig,"06_crossed_detail_control")


## 7. Save raw fields, patterns and provenance

The NPZ archive preserves the complex `exit_wave`, `fft_wave`, magnetization and both analyzer/control intensity stacks. `labels` defines the analyzer axis order. `detector_qx_rad_m` and `detector_qy_rad_m` record the detector coordinates. No analyzer case is independently intensity-normalized in the saved data.

For quantitative use, refine the sample pitch, enlarge the illumination window, halve slice thickness and refine detector footprints. The source window should capture the beam; zero padding cannot recover an illumination field that was cut off. This normal-incidence Jones model and ideal analyzer do not establish experimental detector response or a full vector wide-angle scattering calculation.


In [ ]:
np.savez_compressed(
    OUT / "co_magnon_analyzer.npz",
    x_nm=x_nm,y_nm=y_nm,magnetization=m,incident_wave=E_in,
    exit_wave=exit_wave,reference_exit_wave=reference_exit,fft_wave=fft_wave,
    q_rad_nm=q_rad_nm,labels=np.asarray(labels),
    analyzer_axes=np.stack(list(axes_pass.values())),
    fft_intensities=np.stack([fft_patterns[label] for label in labels]),
    detector_intensities=np.stack([patterns[label] for label in labels]),
    detector_control_intensities=np.stack([reference_patterns[label] for label in labels]),
    detector_x_m=layout.detx,detector_y_m=layout.dety,
    detector_qx_rad_m=layout.detqx,detector_qy_rad_m=layout.detqy,
    optical_channels=optical_channels,
)
notebook_path=ROOT / "tutorials/18_co_magnon_linear_analyzer.ipynb"
source_files=list((ROOT / "src/scattering_calculator").rglob("*.py"))+[notebook_path]
tables=list((ROOT / "src/scattering_calculator/database/material_parameter/refractive_indexes/Co").glob("*.txt"))
provenance={
    "parameters":experiment.to_dict(),"metrics":metrics,
    "python":platform.python_version(),"numpy":np.__version__,"matplotlib":matplotlib.__version__,
    "git_commit":subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip(),
    "git_status":subprocess.check_output(["git","status","--short"],cwd=ROOT,text=True),
    "sha256":{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest()
              for p in source_files+tables},
}
(OUT / "provenance.json").write_text(json.dumps(provenance,indent=2)+"\n")
print("Figures, raw complex fields and provenance saved to:",OUT)
